# Salience network — verb-gen DLD/HSL/TD analyses

Three arms for the **salience network** (frontostriatal FC + salience network size), assembled from the validated `stats/` scripts:

1. between-group **connectivity comparison** (Welch ANOVA),
2. **group x FC** predicting SDQ-emotional (NB, 95% CI),
3. **group x network size** predicting SDQ-emotional (NB, 95% CI).

Each code cell is self-contained (re-imports, reloads data) and also writes its PNG/CSV outputs to `results/` exactly as the scripts do. Run top-to-bottom, or any section on its own. Use the project venv as the kernel.

In [ ]:
%matplotlib inline
# figures render inline AND are still saved to results/ by each script

## 1. Between-group connectivity comparison (Welch ANOVA)

Per-tile 3-group omnibus (DLD/HSL/TD) on the 9 frontostriatal FC tiles, Fisher-z, with BH-FDR across tiles and protected Games-Howell / Dunn post-hoc.

*Source: `stats/stats_group_connectivity.py` (adapted for inline display).*

In [ ]:
"""
stats_group_connectivity.py

Between-group comparison of the 9 frontostriatal FC tiles (3 subcortical x 3
cortical) across the three verb-gen groups (DLD, HSL, TD) using the per-subject
matrices from connectivity/2_run_subject_connectivity_analysis_cab-np.py
(<sub>_03_frontostriatal_FC_3_rest_antstri.csv).

Design (n = 144: DLD=53, HSL=27, TD=64)
---------------------------------------
1. Gather every analysed subject's 3x3 frontostriatal r-matrix, melt to long,
   Fisher-z each tile (z = arctanh r). Long table written as a byproduct.
2. Per tile (9 edges), OMNIBUS on Fisher-z:
     - Welch's ANOVA (does NOT assume equal variance -> robust to DLD's larger
       spread + the unequal n; the 3-group generalisation of the Welch t used
       in the 2-group version). Reports F, p, partial eta^2, and (classically
       computed) omega^2.
     - Kruskal-Wallis as a rank-based robustness backup.
     - Benjamini-Hochberg FDR across the 9 tiles on the Welch omnibus p.
3. Protected POST-HOC (only interpreted where the tile's omnibus p<.05 --
   Fisher-protected logic, valid for 3 groups):
     - Games-Howell (unequal-variance pairwise, Welch-consistent): DLD-TD,
       HSL-TD, DLD-HSL with Hedges g.
     - Dunn (Holm) as the rank-based backup.
   Games-Howell/Dunn are computed for all tiles but flagged `protected` so the
   interpretation respects the gating.
4. Figures:
     - group_frontostriatal_mean.png    : 3 descriptive panels (DLD, HSL, TD),
       tile = group mean r +/- SD (descriptive, in r).
     - group_frontostriatal_omnibusF.png: omnibus Welch F per tile, stars = p,
       bold box = survives BH-FDR.
     - group_frontostriatal_pairwise_g.png : 3 panels (DLD-TD, HSL-TD, DLD-HSL),
       tile = Hedges g (Games-Howell), stars = GH p; tiles whose omnibus is not
       significant are greyed (post-hoc not licensed there).

Test on Fisher-z, display r: r is the interpretable unit, but its sampling
variance depends on the true value, so inference is on z and only means are r.

Input:  results/connectivity_outputs/sub-*/sub-*_03_frontostriatal_FC_3_rest_antstri.csv
        dat_verbgen_analysis_144.csv  (code -> group)
Output: results/connectivity_outputs/group_frontostriatal_long.csv
        results/connectivity_outputs/group_frontostriatal_omnibus.csv
        results/connectivity_outputs/group_frontostriatal_posthoc.csv
        results/connectivity_outputs/group_frontostriatal_mean.png
        results/connectivity_outputs/group_frontostriatal_omnibusF.png
        results/connectivity_outputs/group_frontostriatal_pairwise_g.png

## Author: Han Wang
"""

import glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm
from scipy import stats
import pingouin as pg
import scikit_posthocs as sp

PROJECT_DIR = "/home/hanwang/Apps/Programming/matlab-proj/PFM_MSHBM_MHVerbGen"
CONN_DIR = f"{PROJECT_DIR}/results/connectivity_outputs"
LISTCSV = ("/home/hanwang/Documents/Data/verb_gen_krishnan/"
           "behavioural_scq_sdq/dat_verbgen_analysis_144.csv")
SUFFIX = "_3_rest_antstri"

SUBCORTICAL = ["NAcc", "Caudate", "Putamen"]   # rows
CORTICAL = ["ACC", "AI", "LPFC"]               # cols
GROUPS = ["DLD", "HSL", "TD"]                  # display order (clinical gradient)
GROUP_COLORS = {"DLD": "#d63031", "HSL": "#2ca02c", "TD": "#0984e3"}
# pairwise contrasts of interest (A - B)
CONTRASTS = [("DLD", "TD"), ("HSL", "TD"), ("DLD", "HSL")]


def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""


def omega_sq(groups):
    """Classic one-way omega^2 effect size from a list of arrays."""
    k = len(groups)
    n = sum(len(g) for g in groups)
    grand = np.concatenate(groups).mean()
    ss_b = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    ss_w = sum(((g - g.mean()) ** 2).sum() for g in groups)
    ss_t = ss_b + ss_w
    df_b = k - 1
    ms_w = ss_w / (n - k)
    denom = ss_t + ms_w
    return (ss_b - df_b * ms_w) / denom if denom > 0 else np.nan


# ============================================================
# 1. Gather per-subject frontostriatal tiles -> long (z-transformed)
# ============================================================
beh = pd.read_csv(LISTCSV)[["code", "group"]].copy()
beh["code"] = beh["code"].astype(str)
code2group = beh.set_index("code")["group"].to_dict()

rows, missing = [], []
for code, group in code2group.items():
    sub = f"sub-{code}"
    hits = glob.glob(f"{CONN_DIR}/{sub}/{sub}_03_frontostriatal_FC{SUFFIX}.csv")
    if not hits:
        missing.append(sub)
        continue
    m = pd.read_csv(hits[0], index_col="subcortical")
    for sc in SUBCORTICAL:
        for ct in CORTICAL:
            r = float(m.loc[sc, ct])
            rows.append(dict(subject=sub, code=code, group=group,
                             subcortical=sc, cortical=ct, edge=f"{sc}-{ct}",
                             r=r, z=np.arctanh(np.clip(r, -0.999999, 0.999999))))

long = pd.DataFrame(rows)
if missing:
    print(f"WARNING: {len(missing)} subjects had no frontostriatal CSV: {missing}")
counts = long.drop_duplicates("subject")["group"].value_counts().to_dict()
print(f"Loaded {long['subject'].nunique()} subjects: "
      + ", ".join(f"{g}={counts.get(g,0)}" for g in GROUPS))
long.to_csv(f"{CONN_DIR}/group_frontostriatal_long.csv", index=False)
print(f"Saved: {CONN_DIR}/group_frontostriatal_long.csv")

# ============================================================
# 2. Per-tile omnibus (Welch ANOVA + Kruskal-Wallis) on Fisher-z
# 3. Per-tile protected post-hoc (Games-Howell + Dunn)
# ============================================================
omni, post = [], []
for sc in SUBCORTICAL:
    for ct in CORTICAL:
        edge = f"{sc}-{ct}"
        d = long[(long.subcortical == sc) & (long.cortical == ct)].copy()
        arrs = [d[d.group == g]["z"].to_numpy() for g in GROUPS]

        wa = pg.welch_anova(data=d, dv="z", between="group").iloc[0]
        F, p_w, np2 = float(wa["F"]), float(wa["p_unc"]), float(wa["np2"])
        H, p_kw = stats.kruskal(*arrs)
        w2 = omega_sq(arrs)
        rec = dict(edge=edge, subcortical=sc, cortical=ct,
                   F=F, p=p_w, eta2=np2, omega2=w2, kw_H=H, kw_p=p_kw)
        for g in GROUPS:
            gr = d[d.group == g]["r"]
            rec[f"mean_r_{g}"] = gr.mean()
            rec[f"sd_r_{g}"] = gr.std(ddof=1)
        omni.append(rec)

        # Games-Howell (all tiles; protection applied at interpretation)
        gh = pg.pairwise_gameshowell(data=d, dv="z", between="group")
        gh_lu = {(a, b): row for (a, b), row in
                 gh.set_index(["A", "B"]).iterrows()}
        dunn = sp.posthoc_dunn(d, val_col="z", group_col="group", p_adjust="holm")
        for A, B in CONTRASTS:
            row = gh_lu.get((A, B)) if (A, B) in gh_lu else gh_lu.get((B, A))
            sign = 1.0 if (A, B) in gh_lu else -1.0   # flip if pingouin ordered (B,A)
            post.append(dict(
                edge=edge, subcortical=sc, cortical=ct, contrast=f"{A}-{B}",
                diff_z=sign * float(row["diff"]),
                hedges=sign * float(row["hedges"]),
                gh_T=sign * float(row["T"]), gh_p=float(row["pval"]),
                dunn_p=float(dunn.loc[A, B])))

omni = pd.DataFrame(omni)
post = pd.DataFrame(post)

# BH-FDR across the 9 tiles on the Welch omnibus p
order = np.argsort(omni["p"].to_numpy())
ranks = np.empty(len(omni), int); ranks[order] = np.arange(1, len(omni) + 1)
q = np.minimum(1, omni["p"].to_numpy() * len(omni) / ranks)
q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
q_full = np.empty_like(q); q_full[order] = q_sorted
omni["p_fdr"] = q_full
omni["omnibus_sig"] = omni["p"] < .05          # gates the protected post-hoc

# tag post-hoc rows with whether their tile's omnibus licenses interpretation
post = post.merge(omni[["edge", "p", "p_fdr", "omnibus_sig"]]
                  .rename(columns={"p": "omnibus_p", "p_fdr": "omnibus_p_fdr"}),
                  on="edge", how="left")
post["protected"] = post["omnibus_sig"]

omni.to_csv(f"{CONN_DIR}/group_frontostriatal_omnibus.csv", index=False)
post.to_csv(f"{CONN_DIR}/group_frontostriatal_posthoc.csv", index=False)
print(f"Saved: {CONN_DIR}/group_frontostriatal_omnibus.csv")
print(f"Saved: {CONN_DIR}/group_frontostriatal_posthoc.csv\n")

print("Per-tile omnibus (Welch ANOVA on Fisher-z):")
print(omni[["edge", "mean_r_DLD", "mean_r_HSL", "mean_r_TD",
            "F", "p", "p_fdr", "eta2", "omega2", "kw_p"]]
      .round(3).to_string(index=False))
n_sig = int((omni["p"] < .05).sum()); n_fdr = int((omni["p_fdr"] < .05).sum())
print(f"\nTiles omnibus p<.05: {n_sig}/9 | survive FDR: {n_fdr}/9")
if n_sig:
    print("\nProtected Games-Howell for omnibus-significant tiles:")
    print(post[post.protected][["edge", "contrast", "diff_z", "hedges",
                                "gh_p", "dunn_p"]].round(3).to_string(index=False))
else:
    print("\nNo tile reaches a significant omnibus; post-hoc not licensed.")


def grid(mapping, sc_list=SUBCORTICAL, ct_list=CORTICAL):
    a = np.full((len(sc_list), len(ct_list)), np.nan)
    for i, sc in enumerate(sc_list):
        for j, ct in enumerate(ct_list):
            a[i, j] = mapping.get(f"{sc}-{ct}", np.nan)
    return a


osmap = omni.set_index("edge")

# ============================================================
# 4a. Descriptive: group mean r (+/- SD), one panel per group
# ============================================================
mean_r = {g: grid({e: osmap.loc[e, f"mean_r_{g}"] for e in osmap.index}) for g in GROUPS}
sd_r = {g: grid({e: osmap.loc[e, f"sd_r_{g}"] for e in osmap.index}) for g in GROUPS}
vmax = np.ceil(np.nanmax([np.abs(mean_r[g]) for g in GROUPS]) * 10) / 10

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
cmap = plt.cm.RdBu_r.copy(); cmap.set_bad("white")
for ax, g in zip(axes, GROUPS):
    im = ax.imshow(mean_r[g], cmap=cmap, vmin=-vmax, vmax=vmax, aspect="equal")
    ax.set_xticks(range(len(CORTICAL))); ax.set_yticks(range(len(SUBCORTICAL)))
    ax.set_xticklabels(CORTICAL, fontweight="bold")
    ax.set_yticklabels(SUBCORTICAL, fontweight="bold")
    ax.set_title(f"{g} (n={counts.get(g,0)})", color=GROUP_COLORS[g], fontweight="bold")
    for i in range(len(SUBCORTICAL)):
        for j in range(len(CORTICAL)):
            c = "white" if abs(mean_r[g][i, j]) > 0.3 else "black"
            ax.text(j, i, f"{mean_r[g][i, j]:+.2f}\n±{sd_r[g][i, j]:.2f}",
                    ha="center", va="center", fontsize=9, color=c)
    ax.set_xlabel("Cortical zone")
axes[0].set_ylabel("Subcortical")
cbar = fig.colorbar(im, ax=axes, fraction=0.03, pad=0.03)
cbar.set_label("Group mean Pearson's r")
fig.suptitle("Frontostriatal FC — group means (tile: mean r ± SD across subjects)",
             fontsize=13)
plt.savefig(f"{CONN_DIR}/group_frontostriatal_mean.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {CONN_DIR}/group_frontostriatal_mean.png")

# ============================================================
# 4b. Omnibus F heatmap: stars = omnibus p, bold box = survives FDR
# ============================================================
Fg = grid({e: osmap.loc[e, "F"] for e in osmap.index})
Pg = grid({e: osmap.loc[e, "p"] for e in osmap.index})
Qg = grid({e: osmap.loc[e, "p_fdr"] for e in osmap.index})
fmax = max(np.nanmax(Fg), 1e-6)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(Fg, cmap="viridis", vmin=0, vmax=fmax, aspect="equal")
ax.set_xticks(range(len(CORTICAL))); ax.set_yticks(range(len(SUBCORTICAL)))
ax.set_xticklabels(CORTICAL, fontweight="bold")
ax.set_yticklabels(SUBCORTICAL, fontweight="bold")
for i in range(len(SUBCORTICAL)):
    for j in range(len(CORTICAL)):
        col = "white" if Fg[i, j] < 0.6 * fmax else "black"
        ax.text(j, i, f"{Fg[i, j]:.2f}{stars(Pg[i, j])}",
                ha="center", va="center", fontsize=12, color=col)
        if Qg[i, j] < 0.05:
            ax.add_patch(Rectangle((j-0.5, i-0.5), 1, 1, fill=False,
                                   edgecolor="red", linewidth=3))
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Welch ANOVA F (on Fisher-z)")
ax.set_xlabel("Cortical zone"); ax.set_ylabel("Subcortical")
ax.set_title("Frontostriatal FC: 3-group omnibus (DLD/HSL/TD)\n"
             "* p<.05 ** p<.01 *** p<.001 (uncorr.);  red box = survives BH-FDR",
             fontsize=11)
plt.savefig(f"{CONN_DIR}/group_frontostriatal_omnibusF.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {CONN_DIR}/group_frontostriatal_omnibusF.png")

# ============================================================
# 4c. Pairwise Hedges g heatmaps (3 contrasts); grey where omnibus n.s.
# ============================================================
pmap = post.set_index(["contrast", "edge"])
gmax = max(np.nanmax(np.abs(post["hedges"])), 1e-6)
norm = TwoSlopeNorm(vmin=-gmax, vcenter=0, vmax=gmax)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, (A, B) in zip(axes, CONTRASTS):
    G = grid({e: pmap.loc[(f"{A}-{B}", e), "hedges"] for e in osmap.index})
    Pp = grid({e: pmap.loc[(f"{A}-{B}", e), "gh_p"] for e in osmap.index})
    Prot = grid({e: float(pmap.loc[(f"{A}-{B}", e), "protected"]) for e in osmap.index})
    im = ax.imshow(G, cmap="RdBu_r", norm=norm, aspect="equal")
    ax.set_xticks(range(len(CORTICAL))); ax.set_yticks(range(len(SUBCORTICAL)))
    ax.set_xticklabels(CORTICAL, fontweight="bold")
    ax.set_yticklabels(SUBCORTICAL, fontweight="bold")
    ax.set_title(f"{A} − {B}", fontsize=12, fontweight="bold")
    for i in range(len(SUBCORTICAL)):
        for j in range(len(CORTICAL)):
            if Prot[i, j] < 0.5:                      # omnibus n.s. -> not licensed
                ax.add_patch(Rectangle((j-0.5, i-0.5), 1, 1, facecolor="white",
                                       alpha=0.6, edgecolor="none"))
                txt, col = f"{G[i, j]:+.2f}", "0.6"
            else:
                txt, col = f"{G[i, j]:+.2f}{stars(Pp[i, j])}", \
                    ("white" if abs(G[i, j]) > 0.6 * gmax else "black")
            ax.text(j, i, txt, ha="center", va="center", fontsize=11, color=col)
    ax.set_xlabel("Cortical zone")
axes[0].set_ylabel("Subcortical")
cbar = fig.colorbar(im, ax=axes, fraction=0.03, pad=0.03)
cbar.set_label("Hedges g (Games–Howell, on Fisher-z)")
fig.suptitle("Frontostriatal FC pairwise contrasts (protected: greyed = omnibus n.s.)\n"
             "red = first group higher, blue = lower;  * p<.05 ** p<.01 *** p<.001 (Games–Howell)",
             fontsize=12)
plt.savefig(f"{CONN_DIR}/group_frontostriatal_pairwise_g.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {CONN_DIR}/group_frontostriatal_pairwise_g.png")


## 2. group x FC -> SDQ-emotional (negative binomial, with 95% CI)

One NB model per tile: `emotional ~ C(group,Treatment('TD')) * FCz_c`. Joint 2-df interaction LR + Freedman-Lane permutation; the scatter now carries the delta-method 95% CI band on each group's predicted mean.

*Source: `stats/stats_connectivity_emotional_nb.py` (adapted for inline display).*

In [ ]:
"""
stats_connectivity_emotional_nb.py

Mood-coupling model matching the salience-size analysis
(stats_emotional_salience_nb_interaction.py): the brain measure is the PREDICTOR
and SDQ-emotional the OUTCOME, one negative-binomial model per frontostriatal
tile, now with the 3-level group factor (TD reference) over all 144 subjects:

    emotional ~ C(group, Treatment('TD')) * FCz_c        [TD = reference]

  * Outcome = SDQ-emotional (bounded 0-10 count, floored in TD) -> negative
    binomial (log link), as for salience size.
  * Predictor = tile FC in Fisher-z (FCz), mean-centred per tile, so the group
    terms are the group-vs-TD emotional gaps at that tile's mean connectivity.
  * Two interaction terms now: [T.DLD]:FCz_c and [T.HSL]:FCz_c (each a
    group-vs-TD difference in the FC->emotional slope). The headline
    "interaction" test is a JOINT likelihood-ratio test of BOTH interaction
    terms (2 df: full vs additive), BH-FDR across the 9 tiles. Exploratory.

Input:  results/connectivity_outputs/group_frontostriatal_long.csv
        dat_verbgen_analysis_144.csv  (emotional, group)
Output: results/connectivity_outputs/connectivity_emotional_nb_stats.csv
        results/connectivity_outputs/connectivity_emotional_nb_interaction.png
        results/connectivity_outputs/connectivity_emotional_nb_scatter.png

## Author: Han Wang
"""

import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
NPERM = 2000

PROJECT_DIR = "/home/hanwang/Apps/Programming/matlab-proj/PFM_MSHBM_MHVerbGen"
CONN_DIR = f"{PROJECT_DIR}/results/connectivity_outputs"
LISTCSV = ("/home/hanwang/Documents/Data/verb_gen_krishnan/"
           "behavioural_scq_sdq/dat_verbgen_analysis_144.csv")

SUBCORTICAL = ["NAcc", "Caudate", "Putamen"]
CORTICAL = ["ACC", "AI", "LPFC"]
GROUPS = ["DLD", "HSL", "TD"]
GROUP_COLORS = {"DLD": "#d63031", "HSL": "#2ca02c", "TD": "#0984e3"}
INTER = {"DLD": "C(group, Treatment('TD'))[T.DLD]:FCz_c",
         "HSL": "C(group, Treatment('TD'))[T.HSL]:FCz_c"}
MAIN = {"DLD": "C(group, Treatment('TD'))[T.DLD]",
        "HSL": "C(group, Treatment('TD'))[T.HSL]"}
CONTRASTS = ["DLD", "HSL"]   # vs TD reference
RNG = np.random.default_rng(0)


def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""


def pois_joint_lr(dd):
    """Joint-interaction LR (deviance drop, full vs additive) under a Poisson
    working model -- the statistic for the Freedman-Lane permutation."""
    f = smf.glm("emotional ~ C(group, Treatment('TD')) * FCz_c",
                data=dd, family=sm.families.Poisson()).fit()
    r = smf.glm("emotional ~ C(group, Treatment('TD')) + FCz_c",
                data=dd, family=sm.families.Poisson()).fit()
    return r.deviance - f.deviance


def freedman_lane_p(d):
    """Distribution-free p for the joint group x FC interaction (permute the
    reduced-model residuals, recompute the Poisson joint-interaction LR)."""
    lr_obs = pois_joint_lr(d)
    red = smf.glm("emotional ~ C(group, Treatment('TD')) + FCz_c",
                  data=d, family=sm.families.Poisson()).fit()
    mu = red.fittedvalues.to_numpy()
    resid = d["emotional"].to_numpy() - mu
    dp = d.copy()
    n_ge = n_val = 0
    for _ in range(NPERM):
        dp["emotional"] = np.clip(np.round(mu + resid[RNG.permutation(len(resid))]), 0, None)
        try:
            n_ge += pois_joint_lr(dp) >= lr_obs
            n_val += 1
        except Exception:
            continue
    return (1 + n_ge) / (1 + n_val)


# ------------------------------------------------------------
# Data: long FC (z) + emotional outcome
# ------------------------------------------------------------
long = pd.read_csv(f"{CONN_DIR}/group_frontostriatal_long.csv")
beh = pd.read_csv(LISTCSV)[["code", "emotional"]].copy()
beh["code"] = beh["code"].astype(str)
long["code"] = long["code"].astype(str)
long = long.merge(beh, on="code", how="left").dropna(subset=["emotional"])
long["emotional"] = long["emotional"].round().astype(int).clip(0, 10)
long = long.rename(columns={"z": "FCz"})
print(f"{long['subject'].nunique()} subjects | groups: "
      + long.drop_duplicates('subject')['group'].value_counts().to_dict().__str__())

# ------------------------------------------------------------
# 9 NB models (one per tile), 3-level group
# ------------------------------------------------------------
res, fits = [], {}
for sc in SUBCORTICAL:
    for ct in CORTICAL:
        d = long[(long.subcortical == sc) & (long.cortical == ct)].copy()
        d["FCz_c"] = d["FCz"] - d["FCz"].mean()
        try:
            full = smf.negativebinomial(
                "emotional ~ C(group, Treatment('TD')) * FCz_c", data=d).fit(disp=0)
            red = smf.negativebinomial(
                "emotional ~ C(group, Treatment('TD')) + FCz_c", data=d).fit(disp=0)
            lr = 2 * (full.llf - red.llf)
            lr_p = stats.chi2.sf(lr, 2)          # 2 df: both interaction terms
            rec = dict(edge=f"{sc}-{ct}", subcortical=sc, cortical=ct,
                       slope_TD=full.params["FCz_c"],
                       slope_DLD=full.params["FCz_c"] + full.params[INTER["DLD"]],
                       slope_HSL=full.params["FCz_c"] + full.params[INTER["HSL"]],
                       inter_beta_DLD=full.params[INTER["DLD"]],
                       inter_z_DLD=full.tvalues[INTER["DLD"]],
                       inter_wald_p_DLD=full.pvalues[INTER["DLD"]],
                       inter_beta_HSL=full.params[INTER["HSL"]],
                       inter_z_HSL=full.tvalues[INTER["HSL"]],
                       inter_wald_p_HSL=full.pvalues[INTER["HSL"]],
                       inter_lr_p=lr_p, alpha=full.params["alpha"],
                       inter_perm_p=freedman_lane_p(d))
            fits[(sc, ct)] = (full, d)
            res.append(rec)
        except Exception as e:
            res.append(dict(edge=f"{sc}-{ct}", subcortical=sc, cortical=ct,
                            inter_lr_p=np.nan))
            print(f"  NB failed for {sc}-{ct}: {e}")

stat = pd.DataFrame(res)
# BH-FDR across the 9 joint-interaction LR p-values
p = stat["inter_lr_p"].to_numpy()
o = np.argsort(p)
ranks = np.empty(len(o), int); ranks[o] = np.arange(1, len(o) + 1)
q = np.minimum(1, p * len(o) / ranks)
q[o] = np.minimum.accumulate(q[o][::-1])[::-1]
stat["inter_lr_p_fdr"] = q
# BH-FDR on the distribution-free permutation p as well
pp = stat["inter_perm_p"].to_numpy()
op = np.argsort(pp)
rp = np.empty(len(op), int); rp[op] = np.arange(1, len(op) + 1)
qp = np.minimum(1, pp * len(op) / rp)
qp[op] = np.minimum.accumulate(qp[op][::-1])[::-1]
stat["inter_perm_p_fdr"] = qp

stat.to_csv(f"{CONN_DIR}/connectivity_emotional_nb_stats.csv", index=False)
print(f"Saved: {CONN_DIR}/connectivity_emotional_nb_stats.csv\n")
print("Per-tile NB  emotional ~ group * FCz  (slopes = FC->emotional, log-count):")
print(stat[["edge", "slope_TD", "slope_DLD", "slope_HSL",
            "inter_lr_p", "inter_lr_p_fdr", "inter_perm_p", "inter_perm_p_fdr"]]
      .round(3).to_string(index=False))
print(f"\nJoint interaction — LR p<.05: {(stat['inter_lr_p']<.05).sum()}/9 "
      f"(FDR {(stat['inter_lr_p_fdr']<.05).sum()}/9)  |  "
      f"permutation p<.05: {(stat['inter_perm_p']<.05).sum()}/9 "
      f"(FDR {(stat['inter_perm_p_fdr']<.05).sum()}/9)")


def grid(col):
    a = np.full((3, 3), np.nan)
    m = stat.set_index("edge")
    for i, sc in enumerate(SUBCORTICAL):
        for j, ct in enumerate(CORTICAL):
            a[i, j] = m.loc[f"{sc}-{ct}", col] if col in m.columns else np.nan
    return a


# ------------------------------------------------------------
# Figure 1: interaction heatmaps, one panel per group-vs-TD contrast
# (Wald z; stars = that term's Wald p; box = tile's joint LR survives FDR)
# ------------------------------------------------------------
Q = grid("inter_lr_p_fdr")
Zs = {g: grid(f"inter_z_{g}") for g in CONTRASTS}
Ps = {g: grid(f"inter_wald_p_{g}") for g in CONTRASTS}
zmax = max(np.nanmax([np.abs(Zs[g]) for g in CONTRASTS]), 1e-6)
norm = TwoSlopeNorm(vmin=-zmax, vcenter=0, vmax=zmax)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
for ax, g in zip(axes, CONTRASTS):
    im = ax.imshow(Zs[g], cmap="RdBu_r", norm=norm, aspect="equal")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(CORTICAL, fontweight="bold")
    ax.set_yticklabels(SUBCORTICAL, fontweight="bold")
    ax.set_title(f"{g} slope − TD slope", fontsize=12, fontweight="bold",
                 color=GROUP_COLORS[g])
    for i in range(3):
        for j in range(3):
            col = "white" if abs(Zs[g][i, j]) > 0.6 * zmax else "black"
            ax.text(j, i, f"{Zs[g][i, j]:+.2f}{stars(Ps[g][i, j])}",
                    ha="center", va="center", fontsize=12, color=col)
            if Q[i, j] < 0.05:
                ax.add_patch(Rectangle((j-0.5, i-0.5), 1, 1, fill=False,
                                       edgecolor="black", linewidth=3))
    ax.set_xlabel("Cortical zone")
axes[0].set_ylabel("Subcortical")
cbar = fig.colorbar(im, ax=axes, fraction=0.03, pad=0.03)
cbar.set_label("interaction Wald z  (group slope − TD slope)")
fig.suptitle("NB: SDQ-emotional ~ group × frontostriatal FC  (TD reference)\n"
             "red = group slope > TD, blue = < TD;  stars = Wald p; "
             "bold box = tile's joint interaction survives BH-FDR", fontsize=11)
plt.savefig(f"{CONN_DIR}/connectivity_emotional_nb_interaction.png",
            dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {CONN_DIR}/connectivity_emotional_nb_interaction.png")

# ------------------------------------------------------------
# Figure 2: 3x3 scatter (x=FC z, y=emotional) + NB predicted-mean curves, 3 groups
# ------------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(11, 9.5), sharey=True)
for i, sc in enumerate(SUBCORTICAL):
    for j, ct in enumerate(CORTICAL):
        ax = axes[i, j]
        if (sc, ct) not in fits:
            ax.set_visible(False); continue
        full, d = fits[(sc, ct)]
        zmean = d["FCz"].mean()
        # delta-method 95% CI on the predicted mean (log-count scale, exp back)
        di = full.model.data.design_info
        kf = len(di.column_names)
        beta = np.asarray(full.params)[:kf]          # alpha is the last param
        cov = np.asarray(full.cov_params())[:kf, :kf]
        for g in GROUPS:
            dg = d[d.group == g]
            yj = dg["emotional"] + RNG.uniform(-0.15, 0.15, len(dg))
            ax.scatter(dg["FCz"], yj, s=22, alpha=0.8, color=GROUP_COLORS[g],
                       edgecolor="k", linewidth=0.3, label=g)
            xs = np.linspace(dg["FCz"].min(), dg["FCz"].max(), 40)
            grid_df = pd.DataFrame({"group": g, "FCz": xs, "FCz_c": xs - zmean})
            X = np.asarray(patsy.dmatrix(di, grid_df))
            eta = X @ beta
            se = np.sqrt(np.einsum("ij,jk,ik->i", X, cov, X))
            ax.fill_between(xs, np.exp(eta - 1.96 * se), np.exp(eta + 1.96 * se),
                            color=GROUP_COLORS[g], alpha=0.13, lw=0)
            ax.plot(xs, np.exp(eta), color=GROUP_COLORS[g], lw=1.8)
        plr = stat.set_index("edge").loc[f"{sc}-{ct}", "inter_lr_p"]
        ax.set_title(f"{sc}-{ct}  (joint int LR p={plr:.3f})", fontsize=9)
        ax.set_ylim(-0.5, 10.5); ax.grid(alpha=0.2)
        if i == 2:
            ax.set_xlabel(f"{ct}\nFC (Fisher z)", fontsize=9)
        if j == 0:
            ax.set_ylabel("SDQ emotional", fontsize=9)
axes[0, 0].legend(fontsize=8, loc="best")
fig.suptitle("NB-predicted SDQ-emotional vs frontostriatal FC, by group "
             "(shaded = 95% CI on predicted mean; points y-jittered)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{CONN_DIR}/connectivity_emotional_nb_scatter.png",
            dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {CONN_DIR}/connectivity_emotional_nb_scatter.png")


## 3. group x Salience network size -> SDQ-emotional (negative binomial)

NB model `emotional ~ C(group,Treatment('TD')) * salience_c` with the predicted mean +/- 95% CI band already built in.

*Source: `stats/stats_emotional_salience_nb_interaction.py` (adapted for inline display).*

In [ ]:
"""
stats_emotional_salience_nb_interaction.py

Negative-binomial model of SDQ-emotional symptoms on Salience-network size with a
group x salience interaction, now with the 3-level group factor (TD reference)
over all 144 subjects:

    emotional ~ C(group, Treatment('TD')) * salience_c      (log link, NB2)

The log link keeps the mean non-negative (handles the floor at 0 without a curve)
and the NB dispersion soaks up the DLD over-dispersion. TD's salience slope is the
Lynch-style positive control (higher salience -> more emotional symptoms in
controls); the two interaction terms test whether DLD and HSL depart from it.

Robustness of the interaction:
  (A) NB joint likelihood-ratio test of BOTH interaction terms (2 df).
  (B) Freedman-Lane permutation using a Poisson working model, statistic = the
      joint interaction LR (deviance drop, full vs additive) -- distribution-free.

Variant selects which MS-HBM set the Salience size comes from (full / icafix).

Input:  results/network_size_<variant>/group_network_size_long.csv (Salience size)
        dat_verbgen_analysis_144.csv                                (emotional)
Output: results/network_size_<variant>/emotional_salience_nb_interaction.csv
        results/network_size_<variant>/emotional_salience_interaction_robustness.csv
        results/network_size_<variant>/emotional_vs_salience_nb.png

## Author: Han Wang
"""

import argparse
import warnings
import numpy as np
import pandas as pd
import patsy
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")

PROJECT_DIR = "/home/hanwang/Apps/Programming/matlab-proj/PFM_MSHBM_MHVerbGen"
LISTCSV = ("/home/hanwang/Documents/Data/verb_gen_krishnan/"
           "behavioural_scq_sdq/dat_verbgen_analysis_144.csv")
GROUPS = ["DLD", "HSL", "TD"]
GROUP_COLORS = {"DLD": "#d63031", "HSL": "#2ca02c", "TD": "#0984e3"}
INTER = {"DLD": "C(group, Treatment('TD'))[T.DLD]:salience_c",
         "HSL": "C(group, Treatment('TD'))[T.HSL]:salience_c"}
MAIN = {"DLD": "C(group, Treatment('TD'))[T.DLD]",
        "HSL": "C(group, Treatment('TD'))[T.HSL]"}
RNG = np.random.default_rng(0)

class _Args:  # notebook shim (was argparse) -- change variant here
    variant = "full"
args = _Args()
NS_DIR = f"{PROJECT_DIR}/results/network_size_{args.variant}"

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
beh = pd.read_csv(LISTCSV)[["code", "emotional"]]
beh["code"] = beh["code"].astype(str)
long = pd.read_csv(f"{NS_DIR}/group_network_size_long.csv")
long["code"] = long["code"].astype(str)
sal = long[long["network_label"] == "Salience"][["code", "group", "network_size_pct"]]
dat = (sal.merge(beh, on="code", how="left")
          .rename(columns={"network_size_pct": "salience"})
          .dropna(subset=["emotional", "salience"]))
dat = dat[dat["group"].isin(GROUPS)].copy()
dat["emotional"] = dat["emotional"].round().astype(int).clip(0, 10)
sal_mean = dat["salience"].mean()
dat["salience_c"] = dat["salience"] - sal_mean
print(f"[{args.variant}] n={len(dat)} | "
      + dat["group"].value_counts().to_dict().__str__())

# ------------------------------------------------------------
# (A) NB interaction model, 3-level group
# ------------------------------------------------------------
F_FULL = "emotional ~ C(group, Treatment('TD')) * salience_c"
F_RED = "emotional ~ C(group, Treatment('TD')) + salience_c"
nb_full = smf.negativebinomial(F_FULL, data=dat).fit(disp=0)
nb_red = smf.negativebinomial(F_RED, data=dat).fit(disp=0)
lr_stat = 2 * (nb_full.llf - nb_red.llf)
lr_p = stats.chi2.sf(lr_stat, 2)              # joint: both interaction terms

sl = {"TD": nb_full.params["salience_c"]}
sl["DLD"] = sl["TD"] + nb_full.params[INTER["DLD"]]
sl["HSL"] = sl["TD"] + nb_full.params[INTER["HSL"]]

print("=" * 70)
print(f"NB: emotional ~ group * salience_c  [TD ref]  (variant={args.variant})")
print("=" * 70)
print("Salience slope (log-count) per group:")
for g in GROUPS:
    print(f"  {g}: {sl[g]:+.4f}  (RR={np.exp(sl[g]):.3f} per +1% cortex)")
print(f"TD slope (positive control) Wald p = {nb_full.pvalues['salience_c']:.4g}")
for g in ["DLD", "HSL"]:
    print(f"  interaction {g}-TD = {nb_full.params[INTER[g]]:+.4f}, "
          f"Wald p = {nb_full.pvalues[INTER[g]]:.4g}")
print(f"Joint interaction LR chi2(2) = {lr_stat:.3f}, p = {lr_p:.4g}")

# ------------------------------------------------------------
# (B) Freedman-Lane permutation (Poisson working model; joint LR statistic)
# ------------------------------------------------------------
def pois_joint_lr(data):
    f = smf.glm(F_FULL, data=data, family=sm.families.Poisson()).fit()
    r = smf.glm(F_RED, data=data, family=sm.families.Poisson()).fit()
    return r.deviance - f.deviance          # = 2*(llf_full - llf_red)

lr_obs = pois_joint_lr(dat)
pois_red = smf.glm(F_RED, data=dat, family=sm.families.Poisson()).fit()
mu_red = pois_red.fittedvalues.to_numpy()
resid = dat["emotional"].to_numpy() - mu_red
NPERM = 2000
dperm = dat.copy()
n_ge = n_valid = 0
for _ in range(NPERM):
    dperm["emotional"] = np.clip(np.round(mu_red + resid[RNG.permutation(len(resid))]), 0, None)
    try:
        lrp = pois_joint_lr(dperm)
    except Exception:
        continue
    n_valid += 1
    n_ge += lrp >= lr_obs
p_perm = (1 + n_ge) / (1 + n_valid)

print("\n" + "-" * 70)
print("ROBUSTNESS — joint group x salience interaction")
print(f"  (A) NB likelihood-ratio        chi2(2) = {lr_stat:5.3f}   p = {lr_p:.4f}")
print(f"  (B) Freedman-Lane permutation  ({n_valid} perms)      p = {p_perm:.4f}")

pd.DataFrame([
    dict(test="NB_joint_LR_chi2_2df", statistic=round(lr_stat, 3), p=round(lr_p, 4)),
    dict(test="Freedman_Lane_permutation", statistic=n_valid, p=round(p_perm, 4)),
]).to_csv(f"{NS_DIR}/emotional_salience_interaction_robustness.csv", index=False)
print(f"Saved: {NS_DIR}/emotional_salience_interaction_robustness.csv")

# ------------------------------------------------------------
# Predicted mean + 95% CI per group (delta method on linear predictor)
# ------------------------------------------------------------
design_info = nb_full.model.data.design_info
k = len(design_info.column_names)
beta = np.asarray(nb_full.params)[:k]
cov = np.asarray(nb_full.cov_params())[:k, :k]


def predict_band(group, n=200):
    d = dat[dat["group"] == group]
    xs = np.linspace(d["salience"].min(), d["salience"].max(), n)
    grid = pd.DataFrame({"group": group, "salience": xs, "salience_c": xs - sal_mean})
    X = np.asarray(patsy.dmatrix(design_info, grid))
    eta = X @ beta
    se = np.sqrt(np.einsum("ij,jk,ik->i", X, cov, X))
    return xs, np.exp(eta), np.exp(eta - 1.96 * se), np.exp(eta + 1.96 * se)


pred_rows = []
fig, ax = plt.subplots(figsize=(7.4, 5.6))
for g in GROUPS:
    d = dat[dat["group"] == g]
    c = GROUP_COLORS[g]
    xs, mu, lo, hi = predict_band(g)
    ax.fill_between(xs, lo, hi, color=c, alpha=0.15, lw=0)
    ax.plot(xs, mu, color=c, lw=2.4,
            label=f"{g} (n={len(d)}): RR={np.exp(sl[g]):.2f}/+1%")
    yj = d["emotional"] + RNG.uniform(-0.12, 0.12, len(d))
    ax.scatter(d["salience"], yj, color=c, edgecolor="k", linewidth=0.3,
               s=34, zorder=3, alpha=0.85)
    for xv, mv, lv, hv in zip(xs, mu, lo, hi):
        pred_rows.append(dict(group=g, salience=round(xv, 4), mean=round(mv, 4),
                              ci_lo=round(lv, 4), ci_hi=round(hv, 4)))

ax.set_xlabel("Salience network size (% cortical surface)")
ax.set_ylabel("SDQ emotional symptoms (0-10)")
ax.set_ylim(-0.5, 10.5)
ax.set_title("Emotional symptoms vs Salience size — NB fit (mean ± 95% CI)\n"
             f"joint group × salience interaction LR p = {lr_p:.3f} "
             f"(variant={args.variant})", fontsize=12)
ax.grid(alpha=0.25); ax.set_axisbelow(True)
ax.legend(title="NB predicted mean", fontsize=9, loc="upper center")
plt.tight_layout()
out_png = f"{NS_DIR}/emotional_vs_salience_nb.png"
plt.savefig(out_png, dpi=200, bbox_inches="tight")
plt.show()

pd.DataFrame(pred_rows).to_csv(
    f"{NS_DIR}/emotional_salience_nb_interaction.csv", index=False)
print(f"\nSaved: {out_png}")
print(f"Saved: {NS_DIR}/emotional_salience_nb_interaction.csv")
